# Ejercicio 2 (remoto) - Buscar con palabras, en Chroma local en docker

**Version remota del ejercicio 2.** Mismo contenido, pero Chroma **no** corre
dentro de Colab: corre en el **Docker local**, expuesto a internet por
un tunel de cloudflare.

> Si el ejercicio 2 embebido ya funcionaba, ^por que esta version?
> Para que veas que **el mismo codigo** sirve contra tu servidor Docker real.
> Solo cambia una linea: `PersistentClient` -> `HttpClient`.

**Atencion:** la URL del tunel es temporal. Si el profesor reinicia cloudflared,
la URL cambia y hay que actualizarla. Pidele la URL actual antes de empezar.

## 0. Configuracion de conexion

Pon aqui la URL publica del Chroma del profesor (sin `https://`).
Esta es la del tunel cloudflare:

In [1]:
# === CONFIG: actualiza esta URL con la que te pase el profesor ===
# https://dave-mitsubishi-bones-greater.trycloudflare.com
HOST_CHROMA = "dave-mitsubishi-bones-greater.trycloudflare.com"
PUERTO = 443          # cloudflare termina en HTTPS puerto 443
SSL = True
TENANT = "default_tenant"
DATABASE = "default_database"
NOMBRE_COLECCION = "conceptos"

print(f"Chroma remoto: https://{HOST_CHROMA}")

Chroma remoto: https://dave-mitsubishi-bones-greater.trycloudflare.com


## 1. Instalar dependencias

Igual que en el ejercicio 2: `chromadb` para hablar con la base y
`sentence-transformers` para el modelo que vectoriza el texto.

In [2]:
!pip install -q chromadb sentence-transformers 2>&1 | tail -n 2
import chromadb
from chromadb.utils import embedding_functions
print("Listo.")

google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Listo.


## 2. Conectar con el Chroma remoto (HttpClient)

Aqui esta el **unico cambio** respecto al ejercicio 2 embebido: en vez de
`PersistentClient`, usamos `HttpClient` apuntando a la URL del tunel.

El modelo de embeddings **se carga en Colab**, no en el servidor. Chroma
solo recibe los vectores ya calculados por HTTP. Por eso seguimos necesitando
`sentence-transformers` aqui.

In [12]:
cliente = chromadb.HttpClient(
    host=HOST_CHROMA,
    port=PUERTO,
    ssl=SSL,
    tenant=TENANT,
    database=DATABASE,
)

# Comprobar que el servidor responde (heartbeat)
hb = cliente.heartbeat()
print(f"Heartbeat: {hb}")
print("Conexion OK con Chroma remoto.")

Heartbeat: 1793740404674445003
Conexion OK con Chroma remoto.


## 3. Cargar el modelo de embeddings (en Colab)

Igual que antes: el modelo convierte texto en un vector de 384 numeros.
Se descarga la primera vez (~120 MB).

In [4]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)
print("Modelo cargado.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado.


## 4. Crear la coleccion (con `embedding_function` y distancia coseno)

Borramos la coleccion si ya existia (para empezar limpios) y la creamos de
nuevo. Si hay otros alumnos conectados al mismo Chroma, **usa un nombre
unico** para tu coleccion (p.ej. `conceptos_tuNombre`) para no pisarlos.

In [5]:
# TIP: cambia 'conceptos' por 'conceptos_tuNombre' si compartes servidor.
try:
    cliente.delete_collection(NOMBRE_COLECCION)
    print(f"Coleccion '{NOMBRE_COLECCION}' anterior borrada.")
except Exception as e:
    print(f"No habia coleccion previa: {e}")

coleccion = cliente.create_collection(
    name=NOMBRE_COLECCION,
    embedding_function=ef,                  # clave: Chroma recibe texto, vectoriza ef
    metadata={"hnsw:space": "cosine"},
)
print(f"Coleccion '{coleccion.name}' creada en Chroma remoto.")

No habia coleccion previa: Collection [conceptos] does not exist
Coleccion 'conceptos' creada en Chroma remoto.


## 5. Cargar datos (solo texto; Chroma calcula el vector via ef)

12 palabras en 3 categorias. El `upsert` envia los textos al servidor;
el cliente (Colab) calcula los embeddings con `ef` y los manda por HTTP.

In [6]:
coleccion.upsert(
    ids=["a1","a2","a3","a4","a5",
         "v1","v2","v3","v4",
         "c1","c2","c3"],
    documents=["perro","gato","caballo","lobo","leon",
               "carro","bicicleta","avion","barco",
               "pan","manzana","leche"],
    metadatas=[{"tipo":"animal"}]*5 +
              [{"tipo":"vehiculo"}]*4 +
              [{"tipo":"comida"}]*3,
)
print(f"Registros guardados en el servidor: {coleccion.count()}")

Registros guardados en el servidor: 12


## 6. Buscar con una palabra

`query_texts` recibe texto: el cliente lo vectoriza y le pide al servidor los
vecinos mas cercanos.

In [7]:
def buscar(q, n=5):
    r = coleccion.query(query_texts=[q], n_results=n)
    print(f"Consulta: '{q}'\n")
    for doc, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0]):
        sim = round((1 - dist) * 100, 1)
        print(f"  {doc:<12} [{meta['tipo']:<8}] similitud: {sim}%")
    print()

buscar("perro")

Consulta: 'perro'

  perro        [animal  ] similitud: 100.0%
  lobo         [animal  ] similitud: 65.0%
  caballo      [animal  ] similitud: 53.2%
  pan          [comida  ] similitud: 46.3%
  leche        [comida  ] similitud: 45.9%



Al buscar `perro`, los primeros resultados no son solo `perro`: tambien
aparecen **gato, lobo, leon, caballo**. El modelo ubica los animales cerca.

## 7. La magia: buscar sin palabras en comun

Descripciones que **no comparten ninguna palabra** con los datos guardados.

In [8]:
buscar("mamífero que ladra")
buscar("vehículo para volar")
buscar("alimento hecho de harina")

Consulta: 'mamífero que ladra'

  lobo         [animal  ] similitud: 71.0%
  perro        [animal  ] similitud: 65.3%
  caballo      [animal  ] similitud: 61.0%
  gato         [animal  ] similitud: 52.4%
  pan          [comida  ] similitud: 47.2%

Consulta: 'vehículo para volar'

  avion        [vehiculo] similitud: 70.5%
  carro        [vehiculo] similitud: 54.2%
  barco        [vehiculo] similitud: 40.1%
  pan          [comida  ] similitud: 34.0%
  leon         [animal  ] similitud: 27.9%

Consulta: 'alimento hecho de harina'

  manzana      [comida  ] similitud: 48.3%
  leche        [comida  ] similitud: 41.8%
  pan          [comida  ] similitud: 29.1%
  lobo         [animal  ] similitud: 19.9%
  caballo      [animal  ] similitud: 16.7%



## 8. Comparar con busqueda exacta (LIKE)

La misma consulta por `LIKE` no devuelve nada:

In [9]:
q = "mamífero que ladra".lower()
exactos = [d for d in ["perro","gato","caballo","lobo","leon",
                       "carro","bicicleta","avion","barco",
                       "pan","manzana","leche"]
           if q in d.lower()]
print(f"Busqueda exacta '{q}': {exactos or 'SIN RESULTADOS'}")

Busqueda exacta 'mamífero que ladra': SIN RESULTADOS


## 9. Ver un registro guardado en el servidor

Para confirmar que los datos quedaron en el Chroma del local (no en Colab):

In [10]:
primero = coleccion.get(limit=1, include=["documents", "embeddings", "metadatas"])
print("Texto:    ", primero["documents"][0])
print("Vector:   ", primero["embeddings"][0][:5], "... (384 numeros)")
print("Metadatos:", primero["metadatas"][0])

Texto:     perro
Vector:    [ 0.20481233 -0.17759714 -0.18733425  0.14696907 -0.15670754] ... (384 numeros)
Metadatos: {'tipo': 'animal'}


## 10. Cierre

- **El codigo es practicamente igual** al ejercicio 2 embebido.
- **Lo unico que cambio:** `PersistentClient(path=...)` -> `HttpClient(host=..., ssl=True, ...)`.
- Los datos **ya no viven en Colab**: viven en tu servidor Docker. Si otro
  alumno abre tu URL y el nombre de tu coleccion, puede ver tus datos.

### Diferencia entre los dos modos

| | Ejercicio 2 (embebido) | Ejercicio 2 (remoto) |
|---|---|---|
| Donde corre Chroma | Dentro de Colab | En el Docker del profesor |
| Donde estan los datos | `/content/...` (se borra al cerrar Colab) | En el servidor Docker (persisten) |
| Cliente | `PersistentClient` | `HttpClient` |
| Modelo de embeddings | En Colab | En Colab (igual) |
| Velocidad | Rapido (local) | Depende de la red y el tunel |
| Lo aprendido | Lo mismo | Lo mismo + tu primer BD "real" |

### Siguiente paso (ejercicio 3)

Lo mismo pero con mas datos (148 registros, 4 idiomas), filtro por metadatos
y una mirada al indice HNSW.